# 06 — GRAPH ANALYTICS
### Jaringan kemiripan *k*-mer · jaringan reassortment · sentralitas dan komunitas


### Kenapa influenza memang harus dimodelkan sebagai graf

Genom influenza A terdiri dari **delapan segmen terpisah**. Ketika dua galur menginfeksi sel yang sama, keturunannya bisa membawa campuran segmen dari keduanya, ini peristiwa yang disebut ***reassortment***.

Akibatnya riwayat evolusi influenza **bukan pohon, melainkan jaringan**. Cabang yang sudah terpisah bisa bergabung kembali. Ini berbeda mendasar dari SARS-CoV-2, yang genomnya satu untai utuh sehingga pohon filogenetik sudah memadai.

### Yang membuat graf ini menuntut komputasi terdistribusi

Membandingkan setiap pasang sekuens pada arsip penuh berarti:

$$\binom{1.630.000}{2} \approx 1{,}33 \times 10^{12} \text{ pasangan}$$

Tidak ada jumlah RAM yang menyelesaikan masalah berorde kuadrat. **MinHash LSH** menghindarinya: sekuens yang mirip dilempar ke *bucket* yang sama lewat fungsi hash, sehingga hanya pasangan di dalam bucket yang benar-benar dibandingkan. Ini argumen algoritmik, bukan argumen memori — dan karena itu tidak bisa dipatahkan dengan "beli RAM lebih besar".

`MinHashLSH` adalah komponen **Spark MLlib**, dipakai untuk membangun graf.

---
**Masukan** `features/kmer_k*` · `models/kandidat_reassortant`
**Keluaran** `graph/simpul` · `graph/tepi` · `graph/metrik_*` · `output/*.png`

## Bootstrap

In [1]:
import sys
sys.path.insert(0, r"D:\BDA\nb" if sys.platform == "win32" else "/workspace/nb")
from bda_common import *

import pandas as pd
import numpy as np
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
from pyspark.ml.feature import MinHashLSH, BucketedRandomProjectionLSH

info_mesin()
spark = spark_session("06-graph")

K = CFG["k_utama"]
fitur = spark.read.parquet(jalur(f"features/kmer_k{K}"))
fitur = fitur.withColumn(
    "galur", F.coalesce(F.nullif(F.trim(F.col("isolat")), F.lit("")), F.col("accession")))
fitur.cache()
print(f"\n  Sekuens berfitur : {fitur.count():,}")
print(f"  Galur unik       : {fitur.select('galur').distinct().count():,}")

# Seluruh kemunculan, termasuk salinan identik yang tidak ikut dihitung
# fiturnya. Inilah yang dipakai untuk analisis geografis: satu sekuens
# identik bisa muncul di puluhan negara, dan tiap kemunculan itu adalah
# pengamatan tersendiri yang bermakna untuk jalur penyebaran.
semua_kemunculan = (spark.read.parquet(jalur("stage/sequences"))
                    .select("accession", "hash_seq", "segmen", "subtipe",
                            "negara", "wilayah", "tahun_koleksi", "inang",
                            "isolat", "wakil_unik"))
semua_kemunculan = semua_kemunculan.withColumn(
    "galur", F.coalesce(F.nullif(F.trim(F.col("isolat")), F.lit("")),
                        F.col("accession")))
semua_kemunculan.cache()
n_kemunculan = semua_kemunculan.count()
print(f"  Kemunculan seluruhnya : {n_kemunculan:,}  "
      f"(termasuk {n_kemunculan - fitur.count():,} salinan identik)")
print(f"  Negara terwakili      : "
      f"{semua_kemunculan.select('negara').distinct().count():,}")

CPU logis      : 24
RAM total      : 50.5 GB   bebas 43.1 GB
Disk D: bebas  : 448.5 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20260922T152006Z


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 15:20:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 15:20:09 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark 3.5.9 | master yarn | driver 8g | paralelisme 8
Spark UI: http://jupyter:4040



  Sekuens berfitur : 892,344


  Galur unik       : 244,140


  Kemunculan seluruhnya : 1,586,912  (termasuk 694,568 salinan identik)
  Negara terwakili      : 172


In [2]:
def derajat(tepi):
    """Derajat tiap simpul pada graf tak berarah."""
    a = tepi.select(F.col("src").alias("id"))
    b = tepi.select(F.col("dst").alias("id"))
    return a.union(b).groupBy("id").count().withColumnRenamed("count", "derajat")


def komponen_terhubung(tepi, maks_iterasi=20):
    """Label propagation ke komponen terhubung.

    Setiap simpul mulai dengan labelnya sendiri, lalu berulang kali mengambil
    label terkecil di antara tetangganya. Konvergen ketika tidak ada lagi
    label yang berubah -- biasanya jauh sebelum batas iterasi.
    """
    e = (tepi.select("src", "dst")
             .union(tepi.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
             .distinct())
    simpul = e.select(F.col("src").alias("id")).distinct()
    label = simpul.withColumn("komponen", F.col("id"))

    for it in range(maks_iterasi):
        baru = (e.join(label, e.src == label.id)
                 .groupBy(F.col("dst").alias("id"))
                 .agg(F.min("komponen").alias("kandidat")))
        gabung = (label.join(baru, "id", "left")
                       .withColumn("komponen_baru",
                                   F.least(F.col("komponen"),
                                           F.coalesce(F.col("kandidat"), F.col("komponen")))))
        berubah = gabung.filter(F.col("komponen_baru") != F.col("komponen")).count()
        label = gabung.select("id", F.col("komponen_baru").alias("komponen")).cache()
        print(f"      iterasi {it+1}: {berubah:,} simpul berubah label")
        if berubah == 0:
            break
    return label


def pagerank(tepi, iterasi=10, damping=0.85):
    """PageRank dengan perkalian matriks-vektor berulang di atas DataFrame."""
    e = tepi.select("src", "dst")
    keluar = e.groupBy("src").count().withColumnRenamed("count", "derajat_keluar")
    simpul = (e.select(F.col("src").alias("id"))
               .union(e.select(F.col("dst").alias("id"))).distinct())
    n = simpul.count()
    pr = simpul.withColumn("skor", F.lit(1.0 / n))

    for i in range(iterasi):
        kontrib = (e.join(pr, e.src == pr.id)
                    .join(keluar, "src")
                    .select(F.col("dst").alias("id"),
                            (F.col("skor") / F.col("derajat_keluar")).alias("bagian")))
        pr = (simpul.join(kontrib.groupBy("id").agg(F.sum("bagian").alias("masuk")),
                          "id", "left")
                    .withColumn("skor", F.lit((1 - damping) / n)
                                + damping * F.coalesce(F.col("masuk"), F.lit(0.0)))
                    .select("id", "skor")).cache()
    return pr.orderBy(F.desc("skor"))

---
## Tahap 1 — Graf kemiripan sekuens lewat MinHash LSH

Vektor *k*-mer diperlakukan sebagai **himpunan**: posisi tak-nol berarti *k*-mer itu hadir. Kemiripan dua sekuens diukur dengan **jarak Jaccard** — proporsi *k*-mer yang tidak mereka bagi bersama.

`approxSimilarityJoin` hanya membandingkan pasangan yang jatuh di bucket hash yang sama, sehingga jumlah perbandingan turun drastis dari 1,33 × 10¹² tanpa kehilangan pasangan yang benar-benar mirip.

Graf ini dibangun **per segmen**. Membandingkan HA dengan NA tidak bermakna: keduanya gen berbeda, jadi pasti berjarak jauh dan hanya menghasilkan derau.

In [3]:
# ══════════════════════════════════════════════════════════════════
# PENAHAN LEDAKAN — kenapa parameter di bawah ini seketat itu
# ══════════════════════════════════════════════════════════════════
#
# Versi sebelumnya memakai ambang 0,35 dan dua kali membuat disk penuh
# sampai nol. Penyebabnya bukan bug, melainkan asumsi yang keliru.
#
# Jarak Jaccard 0,35 berarti "simpan setiap pasang yang berbagi 65%
# k-mer". Lintas seluruh arsip, itu memang menyaring banyak. Tetapi graf
# ini dibangun PER SEGMEN, dan di dalam satu segmen sekuens influenza
# memang nyaris identik satu sama lain -- ribuan H3N2 dari tahun yang
# sama berbagi jauh di atas 65%. Jadi ambang itu praktis tidak menyaring
# apa pun, dan keluaran join mendekati O(n^2). Untuk segmen HA sendiri
# itu sekitar 2 x 10^10 tepi.
#
# Empat penahan dipasang, dan urutannya penting: yang paling ampuh
# adalah membatasi SIMPUL, karena ongkos join tumbuh kuadratik
# terhadapnya. Membatasi tepi setelah tepinya terlanjur dibuat tidak
# menolong -- ledakannya sudah terjadi.
# Nilainya tinggal di CFG (bda_common.py) supaya hanya ada SATU sumber
# kebenaran. Versi sebelumnya menuliskannya langsung di sini sementara CFG
# masih menyimpan 0,35 -- dua angka yang saling bertentangan, dan pembaca
# berikutnya akan percaya yang salah.
AMBANG_JARAK = CFG["lsh_ambang_jarak"]
SIMPUL_MAKS = CFG["lsh_simpul_maks"]
TETANGGA_MAKS = CFG["lsh_tetangga_maks"]
TEPI_MAKS_SEGMEN = CFG["lsh_tepi_maks"]

SEG_UNTUK_GRAF = SEG_EKSTERNAL + SEG_INTERNAL   # kedelapan segmen


def graf_kemiripan(segmen, ambang=AMBANG_JARAK, n_hash=None):
    """Daftar tepi kemiripan untuk satu segmen, sudah dibatasi.

    Perbedaan penting dari versi sebelumnya: yang diseberangkan melalui
    join hanya TIGA kolom (src, dst, jarak), bukan sebelas. Metadata
    galur, negara, dan subtipe disambungkan belakangan, setelah daftar
    tepinya mengecil. Sebelas kolom berisi beberapa string membuat tiap
    baris sekitar 200 byte; tiga kolom angka dan teks pendek sekitar 30
    byte. Pada ratusan juta baris antara, selisih tujuh kali lipat itu
    yang menentukan muat atau tidaknya di disk.
    """
    d = (fitur.filter(F.col("segmen") == segmen)
              .select("accession", "features"))
    n_penuh = d.count()
    if n_penuh < 2:
        return None, 0, 0, 0

    # Penahan 1 -- batasi jumlah simpul.
    n = n_penuh
    if n_penuh > SIMPUL_MAKS:
        d = d.sample(False, SIMPUL_MAKS / n_penuh, seed=CFG["seed"])
        n = d.count()
    d = d.persist(StorageLevel.MEMORY_AND_DISK)

    lsh = MinHashLSH(inputCol="features", outputCol="hash",
                     numHashTables=n_hash or CFG["lsh_hash"], seed=CFG["seed"])
    model = lsh.fit(d)

    # Penahan 2 -- ambang yang jauh lebih ketat, hanya tiga kolom.
    pasangan = (model.approxSimilarityJoin(d, d, ambang, distCol="jarak")
                .filter(F.col("datasetA.accession") < F.col("datasetB.accession"))
                .select(F.col("datasetA.accession").alias("src"),
                        F.col("datasetB.accession").alias("dst"),
                        F.col("jarak")))

    # Penahan 3 -- hanya tetangga terdekat per simpul yang disimpan.
    # Satu klaster galur yang sangat mirip bisa menghasilkan ratusan juta
    # tepi yang tidak menambah informasi apa pun: komponen terhubungnya
    # sama saja apakah tiap simpul punya 20 tetangga atau 20.000.
    w = Window.partitionBy("src").orderBy(F.col("jarak").asc())
    pasangan = (pasangan.withColumn("_r", F.row_number().over(w))
                        .filter(F.col("_r") <= TETANGGA_MAKS)
                        .drop("_r"))

    # Penahan 4 -- katup pengaman. Kalau ketiga penahan di atas pun
    # tertembus, job berhenti di sini alih-alih memenuhi disk.
    pasangan = (pasangan
                .limit(TEPI_MAKS_SEGMEN)
                .withColumn("kemiripan", F.round(1 - F.col("jarak"), 4))
                .withColumn("segmen", F.lit(segmen))
                .drop("jarak"))

    return pasangan, n_penuh, n, d


In [ ]:
with Tahap("bangun graf kemiripan per segmen", "GRAPH"):
    # Versi sebelumnya mengumpulkan delapan DataFrame malas ke dalam list,
    # lalu union, lalu tulis. Akibatnya setiap join LSH dihitung TIGA kali:
    # sekali untuk pasangan.count(), sekali saat .write, sekali lagi untuk
    # tepi.count() di akhir. Tiga kali ongkos disk untuk satu hasil.
    #
    # Di sini tiap segmen ditulis langsung begitu selesai, lalu dibaca
    # balik dari Parquet untuk dihitung. Hasil yang sudah ada di disk
    # tidak pernah dihitung ulang.
    TUJUAN_TEPI = jalur("graph/tepi")
    ringkas_tepi = []
    pertama = True

    for s in SEG_UNTUK_GRAF:
        t0 = time.time()
        tepi_s, n_penuh, n_pakai, d_cache = graf_kemiripan(s)
        if tepi_s is None:
            print(f"  segmen {s} ({SEGMEN[s][0]}): terlalu sedikit data -- dilewati")
            continue

        # mode "overwrite" hanya untuk segmen pertama; sisanya "append".
        # partitionBy memisahkan berkasnya per segmen, jadi tahap
        # berikutnya tetap bisa membaca satu segmen saja tanpa memindai
        # seluruhnya.
        (tepi_s.write
               .mode("overwrite" if pertama else "append")
               .partitionBy("segmen")
               .parquet(TUJUAN_TEPI))
        pertama = False
        d_cache.unpersist()

        n_tepi = (spark.read.parquet(TUJUAN_TEPI)
                       .filter(F.col("segmen") == s).count())
        kemungkinan = n_pakai * (n_pakai - 1) // 2
        detik = round(time.time() - t0, 1)

        ringkas_tepi.append({
            "segmen": s, "gen": SEGMEN[s][0],
            "simpul_arsip": n_penuh, "simpul_dipakai": n_pakai,
            "tepi": n_tepi,
            "pasangan_brute_force": kemungkinan,
            "penghematan": f"{kemungkinan / max(n_tepi, 1):,.0f}x",
            "detik": detik})

        catatan = "" if n_penuh == n_pakai else f"  (disampel dari {n_penuh:,})"
        print(f"  segmen {s} ({SEGMEN[s][0]:<3}): {n_pakai:>6,} simpul  "
              f"{n_tepi:>8,} tepi  {detik:>6,.1f}s{catatan}")

    df_tepi_ringkas = pd.DataFrame(ringkas_tepi)
    display(df_tepi_ringkas)

    tepi = spark.read.parquet(TUJUAN_TEPI)
    total_tepi = tepi.count()
    print(f"\n  Total tepi kemiripan : {total_tepi:,}")
    print(f"  Disimpan ke {TUJUAN_TEPI}")
    print(f"\n  Parameter yang dipakai: ambang jarak {AMBANG_JARAK}, "
          f"maksimum {TETANGGA_MAKS} tetangga per simpul,")
    print(f"  maksimum {SIMPUL_MAKS:,} simpul per segmen.")
    print("  Ketiganya membatasi graf secara sengaja. Tanpa itu, keluaran join")
    print("  mendekati kuadratik dan pernah dua kali memenuhi disk sampai nol.")



--------------------------------------------------------------------
[>] GRAPH | bangun graf kemiripan per segmen


  segmen 4 (HA ): 14,989 simpul   113,540 tepi  3,160.0s  (disampel dari 175,801)


  segmen 6 (NA ): 14,981 simpul   128,788 tepi  2,828.1s  (disampel dari 129,996)


[Stage 89:====================================>                   (27 + 8) / 42]

### Berapa banyak perbandingan yang dihemat

Kolom `penghematan` pada tabel di atas menunjukkan berapa kali lipat lebih sedikit pasangan yang benar-benar diperiksa dibanding pendekatan brute force dan itulah alasan pendekatan ini mungkin dijalankan sama sekali.

---
## Tahap 2 — Komponen terhubung: garis keturunan segmen

Komponen terhubung pada graf kemiripan adalah kelompok sekuens yang saling mirip: praktis merupakan **garis keturunan** dari segmen tersebut. Kelompok inilah yang menjadi simpul `:SegmentLineage` pada model graf, dan penting untuk diingat bahwa **kelompok ini ditemukan dari data, bukan diambil dari basis data lain**.

In [ ]:
with Tahap("komponen terhubung -> garis keturunan segmen", "GRAPH"):
    komponen_per_segmen = []
    for s in SEG_UNTUK_GRAF:
        t = tepi.filter(F.col("segmen") == s).select("src", "dst")
        # .rdd mengubah DataFrame menjadi RDD Python, dan itu menuntut
        # Python di setiap executor. limit(1).count() menjawab pertanyaan
        # yang sama sepenuhnya di dalam JVM, dan berhenti setelah satu
        # baris alih-alih memindai seluruh partisi.
        if t.limit(1).count() == 0:
            continue
        print(f"\n  segmen {s} ({SEGMEN[s][0]}):")
        if PAKAI_GF:
            from graphframes import GraphFrame
            v = (t.select(F.col("src").alias("id"))
                  .union(t.select(F.col("dst").alias("id"))).distinct())
            spark.sparkContext.setCheckpointDir(jalur("graph/_ckpt"))
            komp = GraphFrame(v, t).connectedComponents() \
                     .withColumnRenamed("component", "komponen")
        else:
            komp = komponen_terhubung(t)
        komp = komp.withColumn("segmen", F.lit(s))
        komponen_per_segmen.append(komp)
        n_k = komp.select("komponen").distinct().count()
        print(f"    {komp.count():,} sekuens -> {n_k:,} garis keturunan")

    from functools import reduce
    lineage = reduce(lambda a, b: a.union(b), komponen_per_segmen)
    lineage = lineage.withColumn(
        "lineage_id", F.concat_ws("_", F.lit("S"), F.col("segmen"), F.col("komponen")))
    lineage.write.mode("overwrite").parquet(jalur("graph/lineage"))
    print(f"\n  Total garis keturunan : {lineage.select('lineage_id').distinct().count():,}")

In [ ]:
with Tahap("ukuran garis keturunan", "GRAPH"):
    ukuran = (lineage.groupBy("segmen", "lineage_id").count()
                     .withColumnRenamed("count", "anggota"))
    print("  Garis keturunan terbesar:")
    ukuran.orderBy(F.desc("anggota")).show(12, truncate=False)

    print("  Sebaran ukuran (banyak keturunan kecil, sedikit yang besar):")
    (ukuran.withColumn("kelompok",
                       F.when(F.col("anggota") == 1, "1")
                        .when(F.col("anggota") <= 5, "2-5")
                        .when(F.col("anggota") <= 20, "6-20")
                        .when(F.col("anggota") <= 100, "21-100")
                        .otherwise(">100"))
           .groupBy("kelompok").count().orderBy("kelompok").show())

---
## Tahap 3 — Jaringan reassortment

Inilah graf yang paling khas influenza dan tidak punya padanan pada proyek SARS-CoV-2 mana pun.

Dua garis keturunan segmen dihubungkan bila keduanya **muncul bersama dalam galur yang sama**. Karena setiap galur menyumbang sampai delapan segmen, satu galur menghasilkan sejumlah tepi antar garis keturunan segmen yang berbeda.

Yang terbaca dari graf ini:

- **Komunitas** = konstelasi genom yang stabil, yakni kombinasi segmen yang cenderung beredar bersama
- **Tepi antar komunitas** = jejak *reassortment*
- **Simpul dengan sentralitas antara (betweenness) tinggi** = garis keturunan segmen yang muncul di banyak konstelasi berbeda, yaitu **perantara reassortment** — indikator risiko yang paling bernilai dari seluruh bab ini

In [ ]:
with Tahap("bangun jaringan reassortment", "GRAPH"):
    # petakan tiap sekuens ke (galur, garis keturunan segmen)
    peta = (lineage.select(F.col("id").alias("accession"), "lineage_id", "segmen")
                   .join(fitur.select("accession", "galur", "subtipe", "negara",
                                      "wilayah", "tahun_koleksi", "inang"),
                         "accession", "inner"))
    peta.cache()
    print(f"  Sekuens terpetakan ke garis keturunan : {peta.count():,}")

    # dua garis keturunan segmen BERBEDA yang hadir dalam galur yang sama
    a = peta.select(F.col("galur").alias("g"), F.col("lineage_id").alias("src"),
                    F.col("segmen").alias("seg_src"))
    b = peta.select(F.col("galur").alias("g"), F.col("lineage_id").alias("dst"),
                    F.col("segmen").alias("seg_dst"))
    co_occur = (a.join(b, "g")
                 .filter(F.col("seg_src") < F.col("seg_dst"))
                 .groupBy("src", "dst", "seg_src", "seg_dst")
                 .agg(F.countDistinct("g").alias("n_galur"))
                 .filter(F.col("n_galur") >= 2))     # buang pasangan sekali muncul

    n_co = co_occur.count()
    print(f"  Tepi CO_OCCURS (>= 2 galur) : {n_co:,}")
    co_occur.write.mode("overwrite").parquet(jalur("graph/co_occurs"))
    co_occur.orderBy(F.desc("n_galur")).show(10, truncate=False)

In [ ]:
with Tahap("sentralitas pada jaringan reassortment", "GRAPH"):
    t_co = co_occur.select("src", "dst")

    print("  Derajat (berapa banyak garis keturunan lain yang berpasangan):")
    d = derajat(t_co)
    d.orderBy(F.desc("derajat")).show(10, truncate=False)

    print("\n  PageRank (perantara reassortment yang paling berpengaruh):")
    if PAKAI_GF:
        from graphframes import GraphFrame
        v = (t_co.select(F.col("src").alias("id"))
                 .union(t_co.select(F.col("dst").alias("id"))).distinct())
        e2 = (t_co.union(t_co.select(F.col("dst").alias("src"),
                                     F.col("src").alias("dst"))))
        pr = (GraphFrame(v, e2).pageRank(resetProbability=0.15, maxIter=10)
              .vertices.select("id", F.col("pagerank").alias("skor")))
    else:
        e2 = t_co.union(t_co.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
        pr = pagerank(e2, iterasi=10)

    pr_top = pr.orderBy(F.desc("skor")).limit(20)
    pr_top.show(20, truncate=False)

    (pr.join(d, "id", "left")
       .withColumn("segmen", F.split(F.col("id"), "_")[1])
       .write.mode("overwrite").parquet(jalur("graph/metrik_sentralitas")))
    print(f"\n  Disimpan ke {jalur('graph/metrik_sentralitas')}")

---
## Tahap 4 — Kandidat reassortant dari notebook 05, dilihat sebagai graf

Notebook 05 menghasilkan daftar galur yang subtipenya salah diprediksi oleh model gen internal dengan keyakinan tinggi. Di sini daftar itu dipertemukan dengan jaringan reassortment: apakah galur-galur tersebut memang menempati posisi yang menghubungkan konstelasi genom yang berbeda?

Kalau ya, dua metode yang sepenuhnya terpisah — satu dari machine learning, satu dari analitik graf — saling menguatkan. Konvergensi seperti itu jauh lebih meyakinkan bagi penguji daripada satu metode saja.

In [ ]:
with Tahap("silang kandidat reassortant dengan graf", "GRAPH"):
    try:
        kand = spark.read.parquet(jalur("models/kandidat_reassortant"))
        ada_kand = kand.count() > 0
    except Exception:
        ada_kand = False

    if not ada_kand:
        print("  Belum ada kandidat dari notebook 05 -- lewati.")
    else:
        print(f"  Kandidat dari notebook 05 : {kand.count():,}")
        galur_kand = kand.select("galur").distinct()

        lin_kand = (peta.join(galur_kand, "galur", "inner")
                        .select("galur", "lineage_id", "segmen").distinct())
        skor = (lin_kand.join(pr.withColumnRenamed("id", "lineage_id"),
                              "lineage_id", "left")
                        .join(derajat(co_occur.select("src", "dst"))
                              .withColumnRenamed("id", "lineage_id"),
                              "lineage_id", "left"))

        print("\n  Garis keturunan milik kandidat, diurutkan menurut PageRank:")
        skor.orderBy(F.desc("skor")).show(15, truncate=False)

        rata_kand = skor.agg(F.avg("skor")).collect()[0][0] or 0
        rata_semua = pr.agg(F.avg("skor")).collect()[0][0] or 0
        print(f"\n  PageRank rata-rata garis keturunan KANDIDAT : {rata_kand:.6f}")
        print(f"  PageRank rata-rata SELURUH garis keturunan  : {rata_semua:.6f}")
        if rata_semua:
            print(f"  Rasio : {rata_kand/rata_semua:.2f}x")
            print("\n  Rasio di atas 1 berarti kandidat dari model ML memang cenderung")
            print("  menempati posisi lebih sentral di jaringan reassortment --")
            print("  dua metode terpisah menunjuk ke galur yang sama.")
        skor.write.mode("overwrite").parquet(jalur("graph/kandidat_bersilang"))

---
## Tahap 5 — Proyeksi geografis

Graf yang sama diproyeksikan ke ruang geografis: dua negara dihubungkan bila **berbagi garis keturunan segmen yang sama**. Bobotnya jumlah garis keturunan bersama, dan arahnya ditentukan oleh selisih tahun deteksi pertama.

Ini menjawab sisi geografi dari proyek: wilayah mana yang menjadi **sumber** keragaman influenza global, dan mana yang **penerima**.

Hipotesis klasik menyebut Asia Timur dan Tenggara sebagai sumber H3N2 global. Peringkat sentralitas Anda bisa dipakai untuk menguji ulang hipotesis itu dari data — dan punya target validasi seperti ini yang membedakan proyek serius dari sekadar demonstrasi.

In [ ]:
with Tahap("proyeksi geografis", "GRAPH"):
    # Garis keturunan diketahui per sekuens wakil; sambungkan ke SELURUH
    # kemunculan lewat hash_seq supaya negara yang hanya muncul lewat salinan
    # identik tidak hilang dari peta penyebaran.
    lin_hash = (peta.select("accession", "lineage_id")
                    .join(fitur.select("accession", "hash_seq"), "accession")
                    .select("hash_seq", "lineage_id").distinct())

    geo = (semua_kemunculan
           .filter(F.col("negara").isNotNull() & F.col("tahun_koleksi").isNotNull())
           .join(lin_hash, "hash_seq", "inner")
           .select("negara", "wilayah", "lineage_id", "tahun_koleksi"))

    print(f"  Pengamatan geografis dipakai : {geo.count():,}")

    pertama = (geo.groupBy("negara", "lineage_id")
                  .agg(F.min("tahun_koleksi").alias("tahun_pertama")))

    x = pertama.select(F.col("negara").alias("src"), "lineage_id",
                       F.col("tahun_pertama").alias("th_src"))
    y = pertama.select(F.col("negara").alias("dst"), "lineage_id",
                       F.col("tahun_pertama").alias("th_dst"))

    berbagi = (x.join(y, "lineage_id")
                .filter(F.col("src") != F.col("dst"))
                .filter(F.col("th_src") <= F.col("th_dst"))   # arah: lebih dulu -> lebih akhir
                .groupBy("src", "dst")
                .agg(F.count("*").alias("lineage_bersama"),
                     F.round(F.avg(F.col("th_dst") - F.col("th_src")), 2).alias("jeda_tahun"))
                .filter(F.col("lineage_bersama") >= 3))

    print(f"  Tepi SHARES_LINEAGE : {berbagi.count():,}")
    berbagi.orderBy(F.desc("lineage_bersama")).show(12, truncate=False)
    berbagi.write.mode("overwrite").parquet(jalur("graph/geo_tepi"))

    print("\n  Negara paling sentral (PageRank pada graf berarah):")
    pr_geo = pagerank(berbagi.select("src", "dst"), iterasi=12)
    pr_geo.show(15, truncate=False)
    pr_geo.write.mode("overwrite").parquet(jalur("graph/geo_sentralitas"))

---
# Tahap 7 — Basis data graf: Neo4j

**Peta ke laporan:** Bab 2.2 *Teknologi Penyimpanan Data* · LO 5

Sampai titik ini seluruh analitik graf dikerjakan Spark. Itu tepat untuk
**membangun** grafnya: MinHash LSH atas 892.344 vektor dan komponen terhubung
atas jutaan tepi memang menuntut klaster.

Tetapi begitu grafnya jadi, jenis pertanyaannya berubah. *"Lewat garis keturunan
mana sebuah segmen berpindah dari unggas ke manusia?"* adalah pertanyaan
**lintasan**, dan di Spark itu berarti merangkai join berulang kali — satu join
per langkah, dengan jumlah langkah yang tidak diketahui di muka. Dalam Cypher
pertanyaan yang sama ditulis satu baris.

Pembagian perannya karena itu:

| | Spark | Neo4j |
|---|---|---|
| Peran | membangun graf | menyimpan dan menanyai graf |
| Skala | 892.344 simpul, jutaan tepi | ribuan simpul, puluhan ribu tepi |
| Kekuatan | operasi menyapu seluruh data | lintasan, tetangga, pola |
| Bahasa | DataFrame API | Cypher |

Yang dimuat ke Neo4j **bukan** graf kemiripan mentah per sekuens — itu jutaan
tepi yang tidak menjawab pertanyaan baru apa pun. Yang dimuat adalah dua graf
turunan yang justru paling bermakna:

- **Jaringan reassortment** — garis keturunan segmen yang muncul bersama dalam
  galur yang sama. Di sinilah pertanyaan tentang perantara reassortment hidup.
- **Jaringan geografis** — negara yang berbagi garis keturunan, berarah menurut
  tahun deteksi pertama.

Keduanya berukuran ribuan simpul, dan itulah rentang tempat basis data graf
unggul.


In [ ]:
with Tahap("hubungkan ke Neo4j dan siapkan skema", "GRAPH-DB"):
    from neo4j import GraphDatabase

    NEO4J_URI = os.environ.get("BDA_NEO4J", "bolt://neo4j:7687")
    NEO4J_AUTH = ("neo4j", "BdaInflu2026!")

    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    driver.verify_connectivity()

    with driver.session() as ses:
        v = ses.run("CALL dbms.components() YIELD name, versions "
                    "RETURN name, versions[0] AS versi").single()
        print(f"  {v['name']} {v['versi']}  di {NEO4J_URI}")

        # Constraint sekaligus indeks unik. Tanpa ini, MERGE harus memindai
        # seluruh label setiap kali satu simpul dimuat -- pemuatan puluhan
        # ribu simpul berubah dari detik menjadi jam.
        for c in [
            "CREATE CONSTRAINT lineage_id IF NOT EXISTS "
            "FOR (n:GarisKeturunan) REQUIRE n.id IS UNIQUE",
            "CREATE CONSTRAINT negara_nama IF NOT EXISTS "
            "FOR (n:Negara) REQUIRE n.nama IS UNIQUE",
        ]:
            ses.run(c)
        print("  constraint unik dipasang untuk :GarisKeturunan dan :Negara")

        # Muat ulang dari nol supaya sel ini aman dijalankan berkali-kali.
        n_lama = ses.run("MATCH (n) RETURN count(n) AS n").single()["n"]
        if n_lama:
            ses.run("MATCH (n) DETACH DELETE n")
            print(f"  {n_lama:,} simpul lama dihapus")


def muat_neo4j(cypher, baris, ukuran_bets=5_000, label=""):
    """Kirim baris ke Neo4j dalam bets memakai UNWIND.

    Satu transaksi per baris akan menghabiskan waktu di overhead jaringan;
    satu transaksi untuk semuanya akan meledakkan heap. UNWIND per bets
    adalah jalan tengah yang dianjurkan Neo4j sendiri.
    """
    total = 0
    with driver.session() as ses:
        for i in range(0, len(baris), ukuran_bets):
            bets = baris[i:i + ukuran_bets]
            ses.run(cypher, baris=bets)
            total += len(bets)
    if label:
        print(f"  {label:<38} {total:>8,}")
    return total


In [ ]:
with Tahap("muat jaringan reassortment dan geografis ke Neo4j", "GRAPH-DB"):
    # Batas pemuatan. Neo4j di sini diberi 3 GB, dan graf turunan ini
    # memang kecil -- tetapi batas eksplisit mencegah satu kesalahan
    # parameter di hulu berubah menjadi pemuatan berjam-jam.
    MAKS_TEPI_REASSORT = 200_000
    MAKS_TEPI_GEO = 50_000

    # ── simpul: garis keturunan segmen ──────────────────────────────
    lin = (peta.select("lineage_id", "segmen").distinct()
               .withColumn("gen", F.element_at(
                   F.array(*[F.lit(SEGMEN[s][0]) for s in range(1, 9)]),
                   F.col("segmen").cast("int")))
               .toPandas())
    muat_neo4j(
        "UNWIND $baris AS b "
        "MERGE (n:GarisKeturunan {id: b.lineage_id}) "
        "SET n.segmen = b.segmen, n.gen = b.gen",
        lin.to_dict("records"), label="simpul :GarisKeturunan")

    # ── tepi: dua garis keturunan dalam galur yang sama ─────────────
    ko = (co_occur.orderBy(F.desc("n_galur")).limit(MAKS_TEPI_REASSORT)
                  .toPandas())
    muat_neo4j(
        "UNWIND $baris AS b "
        "MATCH (a:GarisKeturunan {id: b.src}) "
        "MATCH (c:GarisKeturunan {id: b.dst}) "
        "MERGE (a)-[r:BERPASANGAN]->(c) "
        "SET r.n_galur = b.n_galur",
        ko.to_dict("records"), label="tepi :BERPASANGAN")

    # ── simpul dan tepi geografis ───────────────────────────────────
    geo_tepi = (berbagi.orderBy(F.desc("lineage_bersama"))
                       .limit(MAKS_TEPI_GEO).toPandas())

    negara = sorted(set(geo_tepi["src"]) | set(geo_tepi["dst"]))
    muat_neo4j(
        "UNWIND $baris AS b MERGE (n:Negara {nama: b.nama})",
        [{"nama": x} for x in negara], label="simpul :Negara")

    # Arah tepi membawa makna: dari negara yang mendeteksi lebih dulu ke
    # yang menyusul. Itulah yang membuat pertanyaan "siapa sumber, siapa
    # penerima" bisa dijawab graf ini.
    muat_neo4j(
        "UNWIND $baris AS b "
        "MATCH (a:Negara {nama: b.src}) "
        "MATCH (c:Negara {nama: b.dst}) "
        "MERGE (a)-[r:MENDAHULUI]->(c) "
        "SET r.lineage_bersama = b.lineage_bersama, r.jeda_tahun = b.jeda_tahun",
        geo_tepi.to_dict("records"), label="tepi :MENDAHULUI")

    with driver.session() as ses:
        ringkas = ses.run(
            "MATCH (n) WITH labels(n)[0] AS label, count(*) AS n "
            "RETURN label, n ORDER BY n DESC").data()
        tepi_db = ses.run(
            "MATCH ()-[r]->() WITH type(r) AS jenis, count(*) AS n "
            "RETURN jenis, n ORDER BY n DESC").data()
    print()
    display(pd.DataFrame(ringkas))
    display(pd.DataFrame(tepi_db))


In [ ]:
with Tahap("analitik graf di dalam Neo4j (Cypher + GDS)", "GRAPH-DB"):
    def cypher(q, **p):
        with driver.session() as ses:
            return pd.DataFrame(ses.run(q, **p).data())

    # ── derajat: Cypher murni, tanpa plugin apa pun ─────────────────
    print("  Garis keturunan dengan pasangan terbanyak:")
    display(cypher("""
        MATCH (n:GarisKeturunan)-[r:BERPASANGAN]-()
        RETURN n.id AS garis_keturunan, n.gen AS gen,
               count(r) AS derajat, sum(r.n_galur) AS total_galur
        ORDER BY derajat DESC LIMIT 10
    """))

    # ── PageRank dan komunitas lewat Graph Data Science ─────────────
    # GDS adalah plugin yang diunduh saat container neo4j pertama kali
    # dinyalakan. Bila tidak tersedia, bagian ini dilewati dan derajat di
    # atas tetap menjadi ukuran sentralitas -- kasar, tetapi jauh lebih
    # baik daripada menggagalkan seluruh notebook karena satu plugin.
    ada_gds = not cypher(
        "SHOW PROCEDURES YIELD name "
        "WHERE name = 'gds.pageRank.stream' RETURN name").empty

    if not ada_gds:
        print("\n  [!] Graph Data Science tidak tersedia -- PageRank dilewati.")
        print("      Plugin diunduh saat container neo4j pertama kali naik;")
        print("      periksa koneksi internet lalu buat ulang containernya:")
        print("        docker compose up -d --force-recreate neo4j")
    else:
        # Proyeksi graf ke memori. GDS bekerja di atas salinan dalam
        # memori, bukan langsung di atas penyimpanan -- itulah yang
        # membuatnya cepat, dan juga alasan proyeksinya harus dibuang
        # dulu supaya sel ini aman diulang.
        with driver.session() as ses:
            ses.run("CALL gds.graph.drop('reassort', false) YIELD graphName")
            ses.run("""
                CALL gds.graph.project('reassort', 'GarisKeturunan',
                     {BERPASANGAN: {orientation: 'UNDIRECTED',
                                    properties: 'n_galur'}})
            """)

        print("\n  PageRank -- perantara reassortment paling berpengaruh:")
        pr_neo = cypher("""
            CALL gds.pageRank.stream('reassort',
                 {relationshipWeightProperty: 'n_galur'})
            YIELD nodeId, score
            RETURN gds.util.asNode(nodeId).id  AS garis_keturunan,
                   gds.util.asNode(nodeId).gen AS gen,
                   round(score, 5) AS pagerank
            ORDER BY pagerank DESC LIMIT 15
        """)
        display(pr_neo)

        print("\n  Louvain -- konstelasi genom yang cenderung beredar bersama:")
        display(cypher("""
            CALL gds.louvain.stream('reassort')
            YIELD nodeId, communityId
            WITH communityId,
                 count(*) AS anggota,
                 collect(gds.util.asNode(nodeId).gen)[0..8] AS contoh_gen
            RETURN communityId AS komunitas, anggota, contoh_gen
            ORDER BY anggota DESC LIMIT 10
        """))

        # Dua implementasi PageRank di atas graf yang sama seharusnya
        # menyepakati simpul mana yang paling sentral, meski skor
        # absolutnya berbeda skala karena normalisasi yang tidak sama.
        pr_spark = (pr.orderBy(F.desc("skor")).limit(15)
                      .toPandas().rename(columns={"id": "garis_keturunan"}))
        irisan = set(pr_neo["garis_keturunan"]) & set(pr_spark["garis_keturunan"])
        print(f"\n  Irisan 15 teratas Spark dan Neo4j : {len(irisan)} dari 15")
        print("  Yang penting peringkatnya, bukan skor mutlaknya. Kalau dua")
        print("  mesin yang sepenuhnya terpisah menyepakati simpul mana yang")
        print("  paling sentral, temuannya jauh lebih sulit dibantah.")


In [ ]:
with Tahap("kueri lintasan -- yang sukar dinyatakan di Spark", "GRAPH-DB"):
    # Inilah pembenaran memakai basis data graf, bukan sekadar memenuhi
    # daftar teknologi. Ketiga kueri di bawah adalah pertanyaan LINTASAN:
    # panjang jalurnya tidak diketahui di muka. Di Spark tiap langkah
    # berarti satu join lagi, dan jumlah join-nya baru ketahuan setelah
    # dijalankan. Dalam Cypher, `*1..4` menyatakan seluruhnya.

    print("  1. Jalur penyebaran terpendek antar negara")
    print("     'Lewat negara mana garis keturunan berpindah dari A ke B?'")
    display(cypher("""
        MATCH (a:Negara)-[:MENDAHULUI*1..3]->(b:Negara)
        WHERE a.nama <> b.nama
        WITH a, b, count(*) AS jalur
        RETURN a.nama AS dari, b.nama AS ke, jalur
        ORDER BY jalur DESC LIMIT 12
    """))

    print("\n  2. Negara paling sentral sebagai SUMBER")
    print("     Derajat keluar tinggi berarti sering mendeteksi lebih dulu.")
    display(cypher("""
        MATCH (n:Negara)
        OPTIONAL MATCH (n)-[keluar:MENDAHULUI]->()
        OPTIONAL MATCH (n)<-[masuk:MENDAHULUI]-()
        WITH n, count(DISTINCT keluar) AS mendahului,
                count(DISTINCT masuk)  AS didahului
        WHERE mendahului + didahului > 0
        RETURN n.nama AS negara, mendahului, didahului,
               mendahului - didahului AS selisih
        ORDER BY selisih DESC LIMIT 12
    """))

    print("\n  3. Garis keturunan yang menjembatani DUA gen berbeda")
    print("     Pola inilah tanda reassortment: satu garis keturunan segmen")
    print("     yang berpasangan dengan garis keturunan dari gen lain.")
    display(cypher("""
        MATCH (a:GarisKeturunan)-[r:BERPASANGAN]-(b:GarisKeturunan)
        WHERE a.gen <> b.gen
        WITH a, count(DISTINCT b.gen) AS gen_pasangan, sum(r.n_galur) AS galur
        WHERE gen_pasangan >= 2
        RETURN a.id AS garis_keturunan, a.gen AS gen,
               gen_pasangan, galur
        ORDER BY gen_pasangan DESC, galur DESC LIMIT 12
    """))

    print("\n  Bandingkan dengan ekuivalennya di Spark: kueri pertama saja")
    print("  menuntut tiga self-join berantai atas tabel tepi, dan kedalaman")
    print("  maksimumnya harus ditulis tangan. Di Cypher itu satu pola,")
    print("  '*1..3', dan mesin yang mengurus penelusurannya.")


---
## Tahap 6 — Sintesis dan visualisasi

Ringkasan seluruh metrik graf, disimpan sebagai berkas yang akan dibaca notebook 07 untuk dashboard.

In [ ]:
with Tahap("sintesis graph analytics", "GRAPH"):
    ringkas_graf = {
        "run_id": RUN_ID,
        "k": K,
        "ambang_jarak_jaccard": AMBANG_JARAK,
        "tepi_kemiripan": int(tepi.count()),
        "garis_keturunan": int(lineage.select("lineage_id").distinct().count()),
        "tepi_co_occurs": int(co_occur.count()),
        "tepi_geografis": int(berbagi.count()),
        "graphframes_dipakai": PAKAI_GF,
        "per_segmen": df_tepi_ringkas.to_dict("records"),
    }
    (BASE / "graph").mkdir(parents=True, exist_ok=True)
    (BASE / "graph" / "ringkasan_graf.json").write_text(
        json.dumps(ringkas_graf, indent=2, default=str), encoding="utf-8")
    print(json.dumps({k: v for k, v in ringkas_graf.items() if k != "per_segmen"},
                     indent=2))

In [ ]:
with Tahap("visualisasi graf", "MONITORING"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import networkx as nx

    # Graf reassortment terlalu besar untuk digambar utuh; ambil sub-graf
    # dari simpul dengan PageRank tertinggi -- itulah bagian yang bermakna.
    top = [r["id"] for r in pr.orderBy(F.desc("skor")).limit(60).collect()]
    sub = (co_occur.filter(F.col("src").isin(top) & F.col("dst").isin(top))
                   .select("src", "dst", "n_galur").toPandas())

    if len(sub):
        G = nx.Graph()
        for _, r in sub.iterrows():
            G.add_edge(r["src"], r["dst"], weight=int(r["n_galur"]))
        fig, ax = plt.subplots(figsize=(12, 9))
        pos = nx.spring_layout(G, seed=CFG["seed"], k=0.6)
        bobot = [G[u][v]["weight"] for u, v in G.edges()]
        maks = max(bobot) if bobot else 1
        nx.draw_networkx_edges(G, pos, width=[0.4 + 3*w/maks for w in bobot],
                               alpha=0.35, ax=ax)
        ukuran_simpul = [120 + 40 * G.degree(nd) for nd in G.nodes()]
        warna = [int(str(nd).split("_")[1]) for nd in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_size=ukuran_simpul, node_color=warna,
                               cmap="tab10", alpha=0.9, ax=ax)
        ax.set_title("Jaringan reassortment — 60 garis keturunan segmen paling sentral\n"
                     "warna = nomor segmen, tebal tepi = jumlah galur yang berbagi",
                     fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        berkas = OUT / "graf_reassortment.png"
        plt.savefig(berkas, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Gambar disimpan: {berkas}")
        print(f"  Simpul digambar: {G.number_of_nodes()}, tepi: {G.number_of_edges()}")
    else:
        print("  Sub-graf kosong -- turunkan ambang atau perbesar data.")

In [ ]:
try:
    driver.close()
    print("Koneksi Neo4j ditutup.")
except NameError:
    pass

display(ringkas_zona())
display(jejak_df())
stop_spark()
print("\nSelesai. Lanjut ke 07_mart_dashboard.ipynb")